In [218]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
import random 

In [219]:
def fix_seeds(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [220]:
n_community = 2
n_members = 3

tokens = []

for ii in range(n_community*n_members+1):
    tokens.append(
        chr(ord('A')+ii)
    )

In [221]:
class RNN1(nn.Module):
    def __init__(self, input_size, hidden_size, hidden_sleep_size, context_size, output_size=7, num_layers=1, modulation_strength=0):
        super(RNN1, self).__init__()

        self.context_fc = nn.Linear(hidden_sleep_size, context_size, bias=False)
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.modulation_strength = modulation_strength

    def set_modulation(self, val):
        self.modulation_strength = val
        
    def forward(self, x, hs, h=None):
        context = self.context_fc(hs)
        # print(hs, context, 'sjs')
        x = torch.cat(((1-self.modulation_strength)*x, self.modulation_strength*context), dim=2)
        # print(x)
        out, h = self.rnn(x, h)
        out = self.fc(out[:,-1,:])

        return out, h

In [222]:
class RNN2(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(RNN2, self).__init__()

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

        self._init_weights_to_zero()

    def forward(self, x, h=None):
        out, h = self.rnn(x, h)
        out = self.fc(out[:,-1,:])

        return out, h
    
    def _init_weights_to_zero(self):
        # Zero out RNN weights and biases
        for name, param in self.rnn.named_parameters():
            nn.init.constant_(param, 0.0)

        # Zero out Linear layer weights and biases
        nn.init.constant_(self.fc.weight, 0.0)
        nn.init.constant_(self.fc.bias, 0.0)

In [223]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=8):
        
        one_hot_encoded = np.zeros((len(data), len(tokens)), dtype=float)
        for ii, token in enumerate(data):
            one_hot_encoded[ii,ord(token)-65] = 1
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory, len(tokens)*working_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), len(tokens)))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                for kk in range(working_memory):
                    self.X[ii,jj,kk*len(tokens):(kk+1)*len(tokens)] = \
                    one_hot_encoded[ii+jj+kk,:]
                    
            self.y[ii] = \
                one_hot_encoded[ii+jj+kk+1,:]

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [202]:
### initial training ###
total_samples = 30000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 40
hidden_sleep_size = 100
context_output_size = 10
num_layers_wake = 1
num_layers_sleep = 1
output_size = len(tokens)
input_size = len(tokens)*working_memory
lr = 1e-3
test_acc = []
c = []
cortical_strength = .5

data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)#gen_seq(total_samples) #

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

network1 = RNN1(input_size+context_output_size, hidden_wake_size, hidden_sleep_size, context_size=context_output_size, output_size=output_size, num_layers=num_layers_wake)
network1.set_modulation(cortical_strength)

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)
hs = torch.zeros((1,1,hidden_sleep_size))
for X, y in train_loader:
    # context = torch.zeros((1,X.size(1),context_output_size))
    # X = torch.cat((X,context), dim=2)
    optimizer.zero_grad()

    if total == 0:
        predicted_y, h = network1(X,hs)
    else:
        predicted_y, h = network1(X, hs, h=mem)
    
    loss = criterion(predicted_y, y)
    
    
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem = h.clone()
        true_y = y.argmax(axis=1)
        estimated_y = predicted_y.argmax(axis=1)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')


Iter : 1001, loss: 2.1501, accuracy: 0.2350
Iter : 2001, loss: 1.8766, accuracy: 0.2500
Iter : 3001, loss: 1.6654, accuracy: 0.3700
Iter : 4001, loss: 3.4640, accuracy: 0.5230
Iter : 5001, loss: 2.0932, accuracy: 0.5600
Iter : 6001, loss: 2.1565, accuracy: 0.6220
Iter : 7001, loss: 1.8559, accuracy: 0.6480
Iter : 8001, loss: 2.1246, accuracy: 0.6340
Iter : 9001, loss: 2.0794, accuracy: 0.6610
Iter : 10001, loss: 2.6737, accuracy: 0.6670
Iter : 11001, loss: 2.0869, accuracy: 0.6520
Iter : 12001, loss: 1.8810, accuracy: 0.6790
Iter : 13001, loss: 1.6867, accuracy: 0.6680
Iter : 14001, loss: 1.7310, accuracy: 0.6830
Iter : 15001, loss: 2.0198, accuracy: 0.6590
Iter : 16001, loss: 1.9462, accuracy: 0.6810
Iter : 17001, loss: 1.9606, accuracy: 0.6660
Iter : 18001, loss: 1.9514, accuracy: 0.6880
Iter : 19001, loss: 1.7801, accuracy: 0.6670
Iter : 20001, loss: 1.9392, accuracy: 0.6560
Iter : 21001, loss: 1.5031, accuracy: 0.6740
Iter : 22001, loss: 1.7073, accuracy: 0.6600
Iter : 23001, loss:

In [217]:
total_samples = 1000000
total_states_to_predict = 5
idx = torch.randint(0, len(tokens), (1,)) [0]
X_hat = torch.zeros(len(tokens),dtype=torch.float32)
X_hat[idx] = 1.0
cortical_strength = .1
SWR = 0
threshold = 0.5
counts = []
seq = ''
train_rnn2 = False

network1.set_modulation(cortical_strength)
hs = torch.randn((1,1,hidden_sleep_size))
network2 = RNN2(hidden_wake_size, hidden_sleep_size, total_states_to_predict*hidden_wake_size, num_layers=num_layers_sleep)
optimizer = torch.optim.SGD(network2.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.MSELoss()

# prev_h = torch.randn((1,1,hidden_wake_size))
target_h = torch.zeros((1,1,total_states_to_predict*hidden_wake_size))
input_h = torch.zeros((1,2,hidden_wake_size))
h_avg = torch.zeros((1,1,hidden_wake_size))
mem = hs.clone()
count = 0
for jj in range(total_samples):
    # fix_seeds()
    with torch.no_grad():
        if jj == 0:
            X_hat_, hidden_state = network1(X_hat.reshape(1,1,-1), hs)
        else:
            # hidden_state += SWR*torch.randn(hidden_state.size())
            X_hat_, hidden_state = network1(X_hat.reshape(1,1,-1), hs, hidden_state)
            if cosine(prev[0][0], hidden_state[0][0]) < threshold:
                train_rnn2 = False
                h_avg += hidden_state.clone()
                count += 1
            else:
                # print('train', cosine(prev[0][0], hidden_state[0][0]), tokens[X_hat.argmax()])
                count += 1
                train_rnn2 = True 
                target_h[0][0][:-hidden_wake_size] = target_h[0][0][hidden_wake_size:].clone()
                target_h[0][0][-hidden_wake_size:] = (h_avg/count).clone()
                input_h[0][0][:] = input_h[0][1][:].clone()
                input_h[0][1][:] = target_h[0][0][2*hidden_wake_size:3*hidden_wake_size]

                h_avg = torch.zeros((1,1,hidden_wake_size))
                count = 0

    ### train RNN2 ###
    if train_rnn2:
        optimizer.zero_grad()
        predicted_h, hs = network2(input_h, mem)
        loss = criterion(predicted_h[0][0], target_h[0][0])
        
        
        loss.backward(retain_graph=True)
        optimizer.step()

    with torch.no_grad():
        prev = hidden_state.clone()
        mem = hs.clone()
        X_hat_prob = torch.nn.functional.softmax(X_hat_, dim=1)
        # fix_seeds(10)
        dist_categ = torch.distributions.Categorical(probs=X_hat_prob.reshape(-1))
        idx = dist_categ.sample()
        # idx = X_hat_prob.argmax()

        # noise = 10*torch.randn(context_output_size)
        X_hat = torch.zeros(len(tokens),dtype=torch.float32)
        # X_hat[len(tokens):] = noise
        X_hat[idx] = 1.0
        X_hat = X_hat.reshape(1,1,-1)
        seq = seq + tokens[idx]
        # prev_h = hidden_state.clone()

        if jj%10000==0:
            print(loss)

        

tensor(0.0776, grad_fn=<MseLossBackward0>)


/opt/anaconda3/envs/efri/lib/python3.12/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([200])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


tensor(0.0672, grad_fn=<MseLossBackward0>)
tensor(0.1002, grad_fn=<MseLossBackward0>)
tensor(0.0939, grad_fn=<MseLossBackward0>)
tensor(0.1333, grad_fn=<MseLossBackward0>)
tensor(0.1034, grad_fn=<MseLossBackward0>)
tensor(0.1045, grad_fn=<MseLossBackward0>)
tensor(0.1159, grad_fn=<MseLossBackward0>)
tensor(0.1071, grad_fn=<MseLossBackward0>)
tensor(0.0898, grad_fn=<MseLossBackward0>)
tensor(0.1081, grad_fn=<MseLossBackward0>)
tensor(0.1037, grad_fn=<MseLossBackward0>)
tensor(0.1579, grad_fn=<MseLossBackward0>)
tensor(0.0891, grad_fn=<MseLossBackward0>)
tensor(0.1302, grad_fn=<MseLossBackward0>)
tensor(0.0975, grad_fn=<MseLossBackward0>)
tensor(0.0805, grad_fn=<MseLossBackward0>)
tensor(0.0765, grad_fn=<MseLossBackward0>)
tensor(0.1462, grad_fn=<MseLossBackward0>)
tensor(0.0932, grad_fn=<MseLossBackward0>)
tensor(0.0669, grad_fn=<MseLossBackward0>)
tensor(0.1262, grad_fn=<MseLossBackward0>)
tensor(0.0864, grad_fn=<MseLossBackward0>)
tensor(0.0690, grad_fn=<MseLossBackward0>)
tensor(0.06

In [216]:
seq

'...'

# Test randomness with noise injection

In [178]:
total_samples = 1000
idx = torch.randint(0, len(tokens), (1,)) [0]
X_hat = torch.zeros(len(tokens),dtype=torch.float32)
X_hat[idx] = 1.0
cortical_strength = .1
SWR = 0
counts = []
seq = ''

network1.set_modulation(cortical_strength)
hs = torch.randn((1,1,hidden_sleep_size))
network2 = RNN2(hidden_wake_size, hidden_sleep_size, hidden_wake_size, num_layers=num_layers_sleep)
optimizer = torch.optim.SGD(network2.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.MSELoss()

prev_h = torch.randn((1,1,hidden_wake_size))
mem = hs.clone()
for jj in range(total_samples):
    # fix_seeds()
    with torch.no_grad():
        if jj == 0:
            X_hat_, hidden_state = network1(X_hat.reshape(1,1,-1), hs)
        else:
            hidden_state += SWR*torch.randn(hidden_state.size())
            X_hat_, hidden_state = network1(X_hat.reshape(1,1,-1), hs, hidden_state)
            print(cosine(prev[0][0], hidden_state[0][0]))

        h_target = hidden_state.clone()

    ### train RNN2 ###
    optimizer.zero_grad()
    predicted_h, hs = network2(prev_h, mem)
    loss = criterion(predicted_h[0][0], h_target[0][0])
    
    
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        prev = hidden_state.clone()
        mem = hs.clone()
        X_hat_prob = torch.nn.functional.softmax(X_hat_, dim=1)
        # fix_seeds(10)
        dist_categ = torch.distributions.Categorical(probs=X_hat_prob.reshape(-1))
        idx = dist_categ.sample()
        # idx = X_hat_prob.argmax()

        # noise = 10*torch.randn(context_output_size)
        X_hat = torch.zeros(len(tokens),dtype=torch.float32)
        # X_hat[len(tokens):] = noise
        X_hat[idx] = 1.0
        X_hat = X_hat.reshape(1,1,-1)
        seq = seq + tokens[idx]
        prev_h = hidden_state.clone()

        if jj%10000==0:
            print(loss)

        

tensor(0.3549, grad_fn=<MseLossBackward0>)
0.4009268923057725
0.9489211581440141
0.8771706949282142
0.19741249647355408
0.30959372448577216
1.0
1.0
0.37612446223427654
0.44502057161443287
1.0
1.0
0.31132125651630116
0.3980566535840785
1.0
0.9203001742701632
0.2137322451796626
0.3254774242913072
1.0
1.0
0.2709319234716725
0.5004038590630258
1.0
0.8341350535495338
0.2701270451259187
0.35750973047479373
0.8186902312360501
0.997423556857445
0.2543997930393146
0.46925899419953365
1.0
0.8378397245783865
0.22396106759032552
0.30859303852594244
0.7700719057685912
0.9986419166453111
0.2632799916941003
0.4867211260082239
1.0
0.9301128216143788
0.2960063540808805
0.3422251825011676
0.9912625331825063
0.8306191521750101
0.32768462099607365
0.5145218131843253
0.9611771680099305
0.8681743788043891
0.38779149540666036
0.5649003972972578
0.9758960985672062
0.8925964691069905
0.3051161857037339
0.42361820912120685
0.8603150478527999
0.8498893748483166
0.2721225341440284
0.36452102525117014
0.8190817426

In [179]:
seq[:200]

'EGCABGBCAGBCAGCABGABCGEFDGABCGEDFGABCGFEDGDEFGDEFGEFDGEFDGBCAGDEFGABCGEFDGFEDGFEDGDEFGEFDGEFDGCBAGABCGFEDGFEDGBCAGFEDGFEDGABCGDFEGDEFGDEFGFEDGABCGCBAGFEDGACBGDEFGCABGBCAGACBGCBAGCBAGACBGBCAGABCGFEDGBA'

In [102]:
total_samples = 1000
idx = torch.randint(0, len(tokens), (1,)) [0]
X_hat = torch.zeros(len(tokens),dtype=torch.float32)
X_hat[idx] = 1.0
cortical_strength = 0.0
SWR = .75
counts = []
seq_ = ''

network1.set_modulation(cortical_strength)
hs = torch.randn((1,1,hidden_sleep_size))
network2 = RNN2(hidden_wake_size, hidden_sleep_size, hidden_wake_size, num_layers=num_layers_sleep)
optimizer = torch.optim.SGD(network2.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.MSELoss()

prev_h = torch.randn((1,1,hidden_wake_size))
mem = hs.clone()
for jj in range(total_samples):
    fix_seeds()
    with torch.no_grad():
        if jj == 0:
            X_hat_, hidden_state = network1(X_hat.reshape(1,1,-1), hs)
        else:
            hidden_state += SWR*torch.randn(hidden_state.size())
            X_hat_, hidden_state = network1(X_hat.reshape(1,1,-1), hs, hidden_state)

        h_target = hidden_state.clone()

    ### train RNN2 ###
    optimizer.zero_grad()
    predicted_h, hs = network2(prev_h, mem)
    loss = criterion(predicted_h[0][0], h_target[0][0])
    
    
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem = hs.clone()
        X_hat_prob = torch.nn.functional.softmax(X_hat_, dim=1)
        fix_seeds(10)
        dist_categ = torch.distributions.Categorical(probs=X_hat_prob.reshape(-1))
        idx = dist_categ.sample()
        # idx = X_hat_prob.argmax()

        # noise = 10*torch.randn(context_output_size)
        X_hat = torch.zeros(len(tokens),dtype=torch.float32)
        # X_hat[len(tokens):] = noise
        X_hat[idx] = 1.0
        X_hat = X_hat.reshape(1,1,-1)
        seq_ = seq_ + tokens[idx]
        prev_h = hidden_state.clone()

        if jj%10000==0:
            print(loss)

        

tensor(0.3817, grad_fn=<MseLossBackward0>)


In [103]:
seq[:100]

'CBGDFEGCBAGDFEGCBAGDFEGCBGCBAGDFEGCBAGDFEGCBAGDFEGDFEGCBAGDFEGCBAGDFEGCBGCBAGDFEGCBAGDFEGCBAGDFEGDFE'

In [104]:
seq_[:400]

'CBGDFGAGBAGBAGBAGACAGBAGCAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGBGAGB'

# Do iterative sleep and wake stages

In [271]:
class brain:
    def __init__(self, input_size, hidden_wake_size, hidden_sleep_size, context_output_size, num_layers_wake, num_layers_sleep, output_size=7, wake_lr=1e-3, sleep_lr=1e-3):
        self.wake_model = RNN1(input_size+context_output_size, hidden_wake_size, hidden_sleep_size, context_size=context_output_size, output_size=output_size, num_layers=num_layers_wake)
        self.wake_optimizer = torch.optim.SGD(self.wake_model.parameters(), lr=wake_lr, momentum=0.95)
        self.wake_criterion = torch.nn.CrossEntropyLoss()

        self.sleep_model = RNN2(hidden_wake_size, hidden_sleep_size, 15*hidden_wake_size, num_layers=num_layers_sleep)
        self.sleep_optimizer = torch.optim.SGD(self.sleep_model.parameters(), lr=sleep_lr, momentum=0.95)
        self.sleep_criterion = torch.nn.MSELoss()

        self.hidden_wake_size = hidden_wake_size
        self.hidden_sleep_size = hidden_sleep_size

    def wake(self, train_loader, cortical_strength=0.5):
        self.wake_model.set_modulation(cortical_strength)
        total = 0
        test_acc = []
        correct = np.zeros(1000,dtype=float)
        hs = torch.zeros((1,1,self.hidden_sleep_size))
        h_avg = torch.zeros((1,1,self.hidden_wake_size))
        count = 0
        for X, y in train_loader:
            self.wake_optimizer.zero_grad()

            if total == 0:
                predicted_y, h = self.wake_model(X,hs)
            else:
                predicted_y, h = self.wake_model(X, hs, h=mem)
            
            loss = self.wake_criterion(predicted_y, y)
            
            
            loss.backward(retain_graph=True)
            self.wake_optimizer.step()

            with torch.no_grad():
                if total>0:
                    if cosine(mem[0][0], h[0][0]) > 0.5:
                        count += 1
                        _, hs = self.sleep_model(h_avg/count, hs)
                        h_avg = torch.zeros((1,1,self.hidden_wake_size))
                        count = 0
                    else:
                        count += 1
                        h_avg += h.clone()

                true_y = y.argmax(axis=1)
                estimated_y = predicted_y.argmax(axis=1)

                mem = h.clone()
                total += 1
                if true_y == estimated_y:
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0

                test_acc.append(
                    np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                )
                if total%1000 == 0:
                    print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')
            
        return test_acc
    def sleep(self, sleep_duration=400000, cortical_strength=0.01, SWR=0.7, threshold=0.5):
        self.wake_model.set_modulation(cortical_strength)
        total_states_to_predict = 15
        idx = torch.randint(0, len(tokens), (1,)) [0]
        X_hat = torch.zeros(len(tokens),dtype=torch.float32)
        X_hat[idx] = 1.0
        cortical_strength = .1
        SWR = 0
        threshold = 0.5
        # counts = []
        seq = ''
        train_rnn2 = False

        hs = torch.randn((1,1,self.hidden_sleep_size))

        # prev_h = torch.randn((1,1,hidden_wake_size))
        target_h = torch.zeros((1,1,total_states_to_predict*self.hidden_wake_size))
        input_h = torch.zeros((1,total_states_to_predict,self.hidden_wake_size))
        h_avg = torch.zeros((1,1,self.hidden_wake_size))
        mem = hs.clone()
        count = 0
        for jj in range(sleep_duration):
            # fix_seeds()
            with torch.no_grad():
                if jj == 0:
                    X_hat_, hidden_state = self.wake_model(X_hat.reshape(1,1,-1), hs)
                else:
                    # hidden_state += SWR*torch.randn(hidden_state.size())
                    X_hat_, hidden_state = self.wake_model(X_hat.reshape(1,1,-1), hs, hidden_state)
                    if cosine(prev[0][0], hidden_state[0][0]) < threshold:
                        train_rnn2 = False
                        h_avg += hidden_state.clone()
                        count += 1
                    else:
                        # print('train', cosine(prev[0][0], hidden_state[0][0]), tokens[X_hat.argmax()])
                        count += 1
                        train_rnn2 = True 
                        target_h[0][0][:-hidden_wake_size] = target_h[0][0][hidden_wake_size:].clone()
                        target_h[0][0][-hidden_wake_size:] = (h_avg/count).clone()
                        input_h = target_h.clone().reshape(1,total_states_to_predict,hidden_wake_size)
                        # input_h[0][:4][:] = input_h[0][1:][:].clone()
                        # input_h[0][4][:] = target_h[0][0][4*hidden_wake_size:5*hidden_wake_size]

                        h_avg = torch.zeros((1,1,hidden_wake_size))
                        count = 0

            ### train RNN2 ###
            if train_rnn2:
                self.sleep_optimizer.zero_grad()
                predicted_h, hs = self.sleep_model(input_h, mem)
                loss = self.sleep_criterion(predicted_h[0][0], target_h[0][0])
                
                
                loss.backward(retain_graph=True)
                self.sleep_optimizer.step()

            with torch.no_grad():
                prev = hidden_state.clone()
                mem = hs.clone()
                X_hat_prob = torch.nn.functional.softmax(X_hat_, dim=1)
                # fix_seeds(10)
                dist_categ = torch.distributions.Categorical(probs=X_hat_prob.reshape(-1))
                idx = dist_categ.sample()
                # idx = X_hat_prob.argmax()

                # noise = 10*torch.randn(context_output_size)
                X_hat = torch.zeros(len(tokens),dtype=torch.float32)
                # X_hat[len(tokens):] = noise
                X_hat[idx] = 1.0
                X_hat = X_hat.reshape(1,1,-1)
                seq = seq + tokens[idx]
                # prev_h = hidden_state.clone()

                if jj%10000==0 and jj>1000:
                    print(loss)

        
    # def sleep(self, sleep_duration=100000, cortical_strength=0.01, SWR=0.7, threshold=0.5):
    #     self.wake_model.set_modulation(cortical_strength)
    #     prev_h = torch.zeros((1,1,2*self.hidden_wake_size))
    #     hs = torch.zeros((1,1,self.hidden_sleep_size))
    #     mem = hs.clone()
    #     idx = torch.randint(0, len(tokens), (1,)) [0]
    #     X_hat = torch.zeros(len(tokens),dtype=torch.float32)
    #     X_hat[idx] = 1.0
    #     for jj in range(sleep_duration):
    #         with torch.no_grad():
    #             if jj == 0:
    #                 X_hat_, hidden_state = self.wake_model(X_hat.reshape(1,1,-1), hs)
    #             else:
    #                 hidden_state += SWR*torch.randn(hidden_state.size())
    #                 X_hat_, hidden_state = self.wake_model(X_hat.reshape(1,1,-1), hs, hidden_state)

    #             h_current = hidden_state.clone()

    #         ### train sleep model ###
    #         self.sleep_optimizer.zero_grad()
    #         predicted_h, hs = self.sleep_model(h_current, mem)
    #         # print(predicted_h[0], h_target[0])
    #         loss = self.sleep_criterion(predicted_h[0], prev_h[0][0])
            
            
    #         loss.backward(retain_graph=True)
    #         optimizer.step()

    #         with torch.no_grad():
    #             mem = hs.clone()
    #             X_hat_prob = torch.nn.functional.softmax(X_hat_, dim=1)
    #             dist_categ = torch.distributions.Categorical(probs=X_hat_prob.reshape(-1))
    #             idx = dist_categ.sample()
                
    #             X_hat = torch.zeros(len(tokens),dtype=torch.float32)
    #             X_hat[idx] = 1.0
    #             X_hat = X_hat.reshape(1,1,-1)
    #             prev_h[0][0][hidden_wake_size:] = prev_h[0][0][:hidden_wake_size]
    #             prev_h[0][0][:hidden_wake_size] = hidden_state.clone()

    #             if jj%10000==0:
    #                 print('Sleep loss iteration ',jj,': ',loss)

In [275]:
# total_samples = 200000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 40
hidden_sleep_size = 10
context_output_size = 2
num_layers_wake = 1
num_layers_sleep = 1
output_size = len(tokens)
input_size = len(tokens)*working_memory
lr = 1e-3
test_acc = []
wake_sleep_cycle = 3


# model = brain(input_size, hidden_wake_size, hidden_sleep_size, context_output_size, num_layers_wake, num_layers_sleep, output_size=output_size, wake_lr=lr, sleep_lr=lr)

for cycle in range(wake_sleep_cycle):
    if cycle == 0:
        total_samples = 40000
    else:
        total_samples = 400000

    data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)

    data_set = Dataset_converter(data, working_memory, short_term_memory)
    train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

    model.wake(train_loader)
    model.sleep()

Iter : 1001, loss: 1.7720, accuracy: 0.6550
Iter : 2001, loss: 1.6603, accuracy: 0.6720
Iter : 3001, loss: 1.8671, accuracy: 0.6670
Iter : 4001, loss: 1.8512, accuracy: 0.6880
Iter : 5001, loss: 1.9509, accuracy: 0.6540
Iter : 6001, loss: 1.7292, accuracy: 0.6770
Iter : 7001, loss: 1.8683, accuracy: 0.6800
Iter : 8001, loss: 1.9236, accuracy: 0.6650
Iter : 9001, loss: 1.9110, accuracy: 0.6760
Iter : 10001, loss: 1.7238, accuracy: 0.6590
Iter : 11001, loss: 1.7102, accuracy: 0.6690
Iter : 12001, loss: 1.8148, accuracy: 0.6760
Iter : 13001, loss: 1.8985, accuracy: 0.6640
Iter : 14001, loss: 1.7851, accuracy: 0.6640
Iter : 15001, loss: 1.7017, accuracy: 0.6670
Iter : 16001, loss: 1.7079, accuracy: 0.6690
Iter : 17001, loss: 1.6678, accuracy: 0.6680
Iter : 18001, loss: 1.7584, accuracy: 0.6710
Iter : 19001, loss: 1.8251, accuracy: 0.6670
Iter : 20001, loss: 1.7354, accuracy: 0.6510
Iter : 21001, loss: 1.7432, accuracy: 0.6750
Iter : 22001, loss: 1.7505, accuracy: 0.6730
Iter : 23001, loss:

In [237]:
o1, h1 = model.wake_model(X, torch.zeros((1,1,model.hidden_sleep_size)))

In [235]:
for X, y in train_loader:
    break

In [236]:
X

tensor([[[0., 0., 1., 0., 0., 0., 0.]]])

In [247]:
o2, h2 = model.wake_model(X, torch.zeros((1,1,model.hidden_sleep_size)))

In [240]:
h1

tensor([[[0.0000, 1.8542, 0.0000, 2.1603, 0.0638, 0.7499, 0.9812, 1.8311,
          0.0000, 0.0000, 0.5693, 0.0000, 0.0000, 0.1537, 0.0000, 0.5149,
          0.2129, 0.0000, 0.8252, 0.0000, 0.6813, 0.0035, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0344, 0.3091, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.2559, 0.7920, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]]],
       grad_fn=<StackBackward0>)

In [248]:
h2

tensor([[[0.0000, 1.8197, 0.0000, 2.3955, 0.0520, 0.0035, 0.9396, 1.8349,
          0.0000, 0.0000, 0.5075, 0.0000, 0.0000, 0.1056, 0.0000, 0.0632,
          0.0762, 0.0000, 0.4018, 0.0429, 0.6112, 0.0000, 0.0000, 0.1579,
          0.0000, 0.0000, 0.0091, 0.1229, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.6761, 0.0258, 0.0000, 0.0000, 0.0000, 0.0000]]],
       grad_fn=<StackBackward0>)

In [249]:
o1

tensor([[ 6.0665,  7.2323, -2.8026, -4.7596, -3.7183, -3.7103,  1.5373]],
       grad_fn=<AddmmBackward0>)

In [250]:
o2

tensor([[ 7.9129,  8.8873, -3.6542, -4.7568, -3.2275, -2.6958, -2.4706]],
       grad_fn=<AddmmBackward0>)

In [251]:
cosine(h1[0][0].detach(), h2[0][0].detach())

0.03826939497790649

In [273]:
input_h

tensor([[[1.0412, 0.0000, 0.0000, 0.0000, 0.5945, 0.0341, 0.0000, 0.0032,
          0.4614, 0.2503, 0.0000, 0.0000, 0.0349, 0.9907, 0.0462, 0.1040,
          0.4445, 0.0000, 0.0000, 0.9447, 0.0344, 0.0139, 0.0000, 0.0000,
          0.8719, 0.0000, 0.4515, 0.8731, 0.0500, 0.0000, 0.7970, 0.0000,
          0.0000, 0.5656, 0.0000, 0.4649, 0.4737, 0.0000, 0.2064, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]]])